In [2]:
import torch
from torch.utils.data import Dataset
from torch.utils.data import DataLoader, Dataset
import pandas as pd

import json
#DataStore store stores the samples and their corresponding labels
#DataLoader wraps an iterable around the Dataset to enable easy access to the samples.

In [3]:
def extract_json(filename: str):
    with open(filename, "r") as f:
        for line in f:
            session = json.loads(line)
            yield session["session"], session["events"]

objects = extract_json("train.jsonl")

print(type(objects))

<class 'generator'>


In [4]:
session = []
aids, timestamps, events_type = [], [], []
for i, (session_id, eventstotal) in enumerate(extract_json("train.jsonl")):
    for event in eventstotal:
        aids.append(event["aid"])
        timestamps.append(event["ts"])
        events_type.append(event["type"])
    if i == 20000:
        break

In [5]:
class OttoDataSet(Dataset): 
    def __init__(self, aid, timestamps, event):
        self.aid = torch.tensor(aid, dtype=torch.int64)
        self.timestamps = torch.tensor(timestamps, dtype=torch.long)
        event_map = {"clicks":1, "carts": 2, "orders": 3}
        self.event = torch.tensor([event_map[e] for e in event], dtype=torch.int64)
        
    def __len__(self) -> int:
        return len(self.aid)
        
    def __getitem__(self, index) -> dict:
        sample = {
            "Aid" : self.aid[index], 
            "TimeStamp": self.timestamps[index], 
            "Event" : self.event[index]}
        return sample
        

In [7]:
dataset = OttoDataSet(aids, timestamps, events_type)
for i in range(0, 5):
    sample = dataset[i]
    print(sample)

{'Aid': tensor(1517085), 'TimeStamp': tensor(1659304800025), 'Event': tensor(1)}
{'Aid': tensor(1563459), 'TimeStamp': tensor(1659304904511), 'Event': tensor(1)}
{'Aid': tensor(1309446), 'TimeStamp': tensor(1659367439426), 'Event': tensor(1)}
{'Aid': tensor(16246), 'TimeStamp': tensor(1659367719997), 'Event': tensor(1)}
{'Aid': tensor(1781822), 'TimeStamp': tensor(1659367871344), 'Event': tensor(1)}
